## Assignment 1:
- Implementing Decision Tree using NumPy.
- Train and evaluate this method on the [Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality) dataset using F1 score.

In [1]:
import numpy as np
import pandas as pd

In [2]:
# =====================================================================
# 1. CẤU TRÚC CÂY QUYẾT ĐỊNH
# =====================================================================

class Node:
    """Nút trong cây quyết định"""
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature       # Chỉ số thuộc tính dùng để phân nhánh
        self.threshold = threshold   # Ngưỡng giá trị để cắt nhánh
        self.left = left             # Nhánh con bên trái (<= threshold)
        self.right = right           # Nhánh con bên phải (> threshold)
        self.value = value           # Nhãn dự đoán (chỉ có ở nút lá)

    def is_leaf_node(self):
        return self.value is not None


class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=10, n_features=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features  
        self.root = None

    def fit(self, X, y):
        # Giới hạn số thuộc tính chọn ngẫu nhiên không vượt quá số lượng thực tế
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
        self.root = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # Điểm dừng thuật toán: đạt độ sâu tối đa, thiếu mẫu hoặc nhánh đã sạch nhãn
        if (depth >= self.max_depth or n_samples < self.min_samples_split or n_labels == 1):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        # Lấy ngẫu nhiên tập chỉ số thuộc tính để tìm điểm cắt dữ liệu
        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        # Dừng phân nhánh nếu tất cả các lượt cắt không làm tăng Information Gain
        if best_feat is None:
            return Node(value=self._most_common_label(y))

        # Chia dữ liệu và đệ quy xây dựng các nhánh con
        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left = self._build_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._build_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(feature=best_feat, threshold=best_thresh, left=left, right=right)

    def _best_split(self, X, y, feat_idxs):
        best_gain = -1e-7  # Đặt mốc âm rất nhỏ để ép IG bắt buộc phải > 0
        split_idx, split_thresh = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for thresh in thresholds:
                gain = self._information_gain(y, X_column, thresh)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thresh

        return split_idx, split_thresh

    def _information_gain(self, y, X_column, threshold):
        parent_entropy = self._entropy(y)

        left_idxs, right_idxs = self._split(X_column, threshold)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r

        return parent_entropy - child_entropy

    def _split(self, X_column, split_thresh):
        left_idxs = np.where(X_column <= split_thresh)[0]
        right_idxs = np.where(X_column > split_thresh)[0]
        return left_idxs, right_idxs

    def _entropy(self, y):
        # Tính toán nhanh bằng vector hóa trên mảng NumPy thay vì chạy vòng lặp
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum(ps[ps > 0] * np.log2(ps[ps > 0]))

    def _most_common_label(self, y):
        if len(y) == 0:
            return 0
        return np.bincount(y).argmax()

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [3]:
# =====================================================================
# 2. HÀM TÍNH F1-SCORE 
# =====================================================================

def calculate_f1_score(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    if precision + recall == 0:
        return 0.0
        
    return 2 * (precision * recall) / (precision + recall)

In [6]:
# =====================================================================
# 3. ĐỌC DỮ LIỆU, TIỀN XỬ LÝ VÀ CHẠY THỬ NGHIỆM
# =====================================================================

if __name__ == "__main__":
    print("--- Đọc và gộp dữ liệu (red và white) từ thư mục 'data/' ---")
    
    df_red = pd.read_csv("data/winequality-red.csv", sep=';')
    df_white = pd.read_csv("data/winequality-white.csv", sep=';')

    # Phân biệt loại rượu: 0 cho vang đỏ, 1 cho vang trắng
    df_red['is_white'] = 0
    df_white['is_white'] = 1

    # Nối 2 tập dữ liệu lại theo dòng
    df = pd.concat([df_red, df_white], ignore_index=True)

    print(f"Số mẫu vang đỏ  : {df_red.shape[0]}")
    print(f"Số mẫu vang trắng: {df_white.shape[0]}")
    print(f"Tổng số mẫu     : {df.shape[0]}")

    # Gán nhãn nhị phân: Điểm chất lượng (quality) > 5 là rượu Tốt (1), ngược lại là Kém (0)
    df['target'] = (df['quality'] > 5).astype(int)
    df = df.drop(columns=['quality'])

    # Trích xuất ma trận thuộc tính X và vector nhãn y
    data = df.to_numpy()
    X = data[:, :-1]
    y = data[:, -1].astype(int)

    # Chia dữ liệu Train / Test theo tỷ lệ 80/20
    np.random.seed(42)  # Cố định seed tổng để đồng nhất kết quả giữa các lần chạy
    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)

    train_size = int(0.8 * len(indices))
    train_indices = indices[:train_size]
    test_indices = indices[train_size:]

    X_train, y_train = X[train_indices], y[train_indices]
    X_test, y_test = X[test_indices], y[test_indices]

    print(f"Số mẫu tập Train: {X_train.shape[0]} | Số mẫu tập Test: {X_test.shape[0]}")
    print("----------------------------------------------------------------")

    # Chuẩn hóa dữ liệu theo Z-score trước khi huấn luyện (Tránh lệch khoảng giá trị đặc trưng)
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    std[std == 0] = 1e-9  # Chặn lỗi chia cho 0 nếu có cột có biến thiên = 0

    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    # Tìm kiếm độ sâu cây (max_depth) tối ưu dựa trên chỉ số F1-Score
    best_f1 = 0
    best_depth = 0
    best_clf = None

    print("--- Chạy thử nghiệm các mốc max_depth... ---")
    for depth in [5, 7, 10, 12, 15, 17, 20]:
        # Cố định seed trước khi fit để quá trình chọn ngẫu nhiên thuộc tính không đổi
        np.random.seed(42) 
        
        clf = DecisionTree(min_samples_split=5, max_depth=depth)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        current_f1 = calculate_f1_score(y_test, y_pred)
        print(f" > Thử max_depth = {depth:2d} | F1-Score = {current_f1:.4f}")
        
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_depth = depth
            best_clf = clf  # Giữ lại mô hình tốt nhất đạt được

    print(f"\n=> max_depth tối ưu: {best_depth} (F1-Score cao nhất = {best_f1:.4f})")
    print("----------------------------------------------------------------")

    # Đánh giá lại lần cuối trên mô hình tối ưu đã lưu
    print("--- Kết quả đánh giá cuối cùng ---")
    y_pred_best = best_clf.predict(X_test)

    final_f1 = calculate_f1_score(y_test, y_pred_best)
    final_accuracy = np.sum(y_test == y_pred_best) / len(y_test)

    print("\n=== KẾT QUẢ ĐÁNH GIÁ DECISION TREE ===")
    print(f"Accuracy  : {final_accuracy * 100:.2f}%")
    print(f"F1-Score  : {final_f1:.4f}")

--- Đọc và gộp dữ liệu (red và white) từ thư mục 'data/' ---
Số mẫu vang đỏ  : 1599
Số mẫu vang trắng: 4898
Tổng số mẫu     : 6497
Số mẫu tập Train: 5197 | Số mẫu tập Test: 1300
----------------------------------------------------------------
--- Chạy thử nghiệm các mốc max_depth... ---
 > Thử max_depth =  5 | F1-Score = 0.7934
 > Thử max_depth =  7 | F1-Score = 0.8017
 > Thử max_depth = 10 | F1-Score = 0.8060
 > Thử max_depth = 12 | F1-Score = 0.8096
 > Thử max_depth = 15 | F1-Score = 0.8093
 > Thử max_depth = 17 | F1-Score = 0.8166
 > Thử max_depth = 20 | F1-Score = 0.8185

=> max_depth tối ưu: 20 (F1-Score cao nhất = 0.8185)
----------------------------------------------------------------
--- Kết quả đánh giá cuối cùng ---

=== KẾT QUẢ ĐÁNH GIÁ DECISION TREE ===
Accuracy  : 77.38%
F1-Score  : 0.8185


## Assignment 2:
- Implementing Random Forest using NumPy.
- Train and evaluate this method on the [Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality) dataset using F1 score.

In [8]:
# =====================================================================
# 1. ĐỊNH NGHĨA CẤU TRÚC NÚT VÀ CÂY QUYẾT ĐỊNH (BASE ESTIMATOR)
# =====================================================================

class Node:
    """Nút trong cây quyết định"""
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature       # Chỉ số thuộc tính dùng để phân nhánh
        self.threshold = threshold   # Ngưỡng giá trị để cắt nhánh
        self.left = left             # Nhánh con bên trái (<= threshold)
        self.right = right           # Nhánh con bên phải (> threshold)
        self.value = value           # Nhãn dự đoán nếu là nút lá

    def is_leaf_node(self):
        return self.value is not None


class DecisionTree:
    """Cây quyết định đơn lẻ làm nền tảng cho Random Forest"""
    def __init__(self, min_samples_split=2, max_depth=10, n_features=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features  
        self.root = None

    def fit(self, X, y):
        # Đảm bảo số lượng thuộc tính lấy ngẫu nhiên không vượt quá thực tế
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
        self.root = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # Điều kiện dừng: Đạt độ sâu tối đa, thiếu mẫu hoặc nút đã thuần nhất
        if (depth >= self.max_depth or n_samples < self.min_samples_split or n_labels == 1):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        # Chọn ngẫu nhiên tập chỉ số thuộc tính (đặc trưng của Random Forest)
        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        # Nếu việc cắt không làm tăng Information Gain thì dừng phân nhánh
        if best_feat is None:
            return Node(value=self._most_common_label(y))

        # Chia dữ liệu và đệ quy xây dựng các nhánh con
        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left = self._build_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._build_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(feature=best_feat, threshold=best_thresh, left=left, right=right)

    def _best_split(self, X, y, feat_idxs):
        best_gain = -1e-7  
        split_idx, split_thresh = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for thresh in thresholds:
                gain = self._information_gain(y, X_column, thresh)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thresh

        return split_idx, split_thresh

    def _information_gain(self, y, X_column, threshold):
        parent_entropy = self._entropy(y)

        left_idxs, right_idxs = self._split(X_column, threshold)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r

        return parent_entropy - child_entropy

    def _split(self, X_column, split_thresh):
        left_idxs = np.where(X_column <= split_thresh)[0]
        right_idxs = np.where(X_column > split_thresh)[0]
        return left_idxs, right_idxs

    def _entropy(self, y):
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum(ps[ps > 0] * np.log2(ps[ps > 0]))

    def _most_common_label(self, y):
        if len(y) == 0:
            return 0
        return np.bincount(y).argmax()

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [9]:
# =====================================================================
# 2. TRIỂN KHAI THUẬT TOÁN RANDOM FOREST
# =====================================================================

class RandomForest:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, n_features=None):
        self.n_trees = n_trees                  # Số lượng cây trong rừng
        self.max_depth = max_depth              # Độ sâu tối đa của mỗi cây
        self.min_samples_split = min_samples_split
        self.n_features = n_features            # Số thuộc tính chọn ngẫu nhiên cho mỗi cây
        self.trees = []                         # Danh sách lưu các thực thể cây

    def _bootstrap_samples(self, X, y):
        """Lấy mẫu ngẫu nhiên có lặp lại (Bootstrapping)"""
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, n_samples, replace=True)
        return X[idxs], y[idxs]

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            # Nếu không chỉ định n_features, mặc định lấy căn bậc hai số lượng thuộc tính (Chuẩn thuật toán RF)
            if self.n_features is None:
                n_feats = int(np.sqrt(X.shape[1]))
            else:
                n_feats = self.n_features
                
            # Khởi tạo một cây quyết định mới
            tree = DecisionTree(max_depth=self.max_depth, 
                                min_samples_split=self.min_samples_split, 
                                n_features=n_feats)
            
            # Lấy mẫu dữ liệu ngẫu nhiên (Bootstrap) và huấn luyện cây
            X_sample, y_sample = self._bootstrap_samples(X, y)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        """Thu thập kết quả từ các cây và bỏ phiếu số đông (Majority Vote)"""
        # Thu thập mảng dự đoán của từng cây: dạng (n_trees, n_samples)
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        
        # Chuyển vị mảng về dạng (n_samples, n_trees) để duyệt theo từng mẫu dữ liệu
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        
        # Bỏ phiếu số đông cho từng dòng mẫu dữ liệu
        predictions = [np.bincount(sample_pred).argmax() for sample_pred in tree_preds]
        return np.array(predictions)


# =====================================================================
# 3. HÀM TÍNH CHỈ SỐ F1-SCORE
# =====================================================================

def calculate_f1_score(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

In [ ]:
# =====================================================================
# 4. ĐỌC DỮ LIỆU CỤC BỘ, TIỀN XỬ LÝ VÀ CHẠY THỰC NGHIỆM RANDOM FOREST
# =====================================================================

if __name__ == "__main__":
    print("--- Đọc và gộp dữ liệu từ thư mục 'data/' ---")
    df_red = pd.read_csv("data/winequality-red.csv", sep=';')
    df_white = pd.read_csv("data/winequality-white.csv", sep=';')
    
    # Bổ sung thuộc tính loại rượu để tăng tính phân loại: 0 cho đỏ, 1 cho trắng
    df_red['is_white'] = 0
    df_white['is_white'] = 1
    
    # Gộp 2 DataFrame lại theo chiều dọc
    df = pd.concat([df_red, df_white], ignore_index=True)

    print(f"Số mẫu vang đỏ  : {df_red.shape[0]}")
    print(f"Số mẫu vang trắng: {df_white.shape[0]}")
    print(f"Tổng số mẫu sau khi gộp: {df.shape[0]}")

    # Chuyển bài toán sang phân loại nhị phân: quality > 5 là Tốt (1), còn lại là Kém (0)
    df['target'] = (df['quality'] > 5).astype(int)
    df = df.drop(columns=['quality'])

    # Trích xuất sang định dạng mảng NumPy
    data = df.to_numpy()
    X = data[:, :-1]
    y = data[:, -1].astype(int)

    # Chia tập dữ liệu Train / Test theo tỷ lệ 80/20
    np.random.seed(42) 
    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)

    train_size = int(0.8 * len(indices))
    train_indices = indices[:train_size]
    test_indices = indices[train_size:]

    X_train, y_train = X[train_indices], y[train_indices]
    X_test, y_test = X[test_indices], y[test_indices]

    # Chuẩn hóa dữ liệu theo phương pháp Z-score
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    std[std == 0] = 1e-9
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    print(f"Mẫu tập Train  : {X_train.shape[0]} | Mẫu tập Test: {X_test.shape[0]}")
    print("----------------------------------------------------------------")

    # Huấn luyện mô hình Random Forest (Ví dụ với cấu hình tối ưu 40 cây, độ sâu tối đa 20, chọn 5 thuộc tính ngẫu nhiên mỗi lần phân nhánh)
    print("--- Đang huấn luyện mô hình Random Forest... ---")
    forest = RandomForest(n_trees=40, max_depth=20, min_samples_split=2, n_features=5)
    forest.fit(X_train, y_train)

    # Dự đoán trên tập kiểm thử
    y_pred = forest.predict(X_test)
    
    # Tính toán các chỉ số đánh giá
    f1 = calculate_f1_score(y_test, y_pred)
    accuracy = np.sum(y_test == y_pred) / len(y_test)

    print("\n=== KẾT QUẢ ĐÁNH GIÁ RANDOM FOREST ===")
    print(f"Độ chính xác (Accuracy): {accuracy * 100:.2f}%")
    print(f"Chỉ số F1-Score        : {f1:.4f}")

--- Đọc và gộp dữ liệu từ thư mục 'data/' ---
Số mẫu vang đỏ  : 1599
Số mẫu vang trắng: 4898
Tổng số mẫu sau khi gộp: 6497
Mẫu tập Train  : 5197 | Mẫu tập Test: 1300
----------------------------------------------------------------
--- Đang huấn luyện mô hình Random Forest... ---

=== KẾT QUẢ ĐÁNH GIÁ RANDOM FOREST ===
Độ chính xác (Accuracy): 83.92%
Chỉ số F1-Score        : 0.8757


## Assignment 3:
- Train and evaluate the Decision Tree method using a machine learning library.
- Train and evaluate the Random Forest method using a machine learning library.

In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

ImportError: DLL load failed while importing _flapack: The paging file is too small for this operation to complete.